# 인증

## 역할(role)

- 권한 위임에 대한 정의
- 구성요소
  - 권한 정책(Permission Policy): 어떤 권한을 위임할 것인가
  - 신뢰 대상(Trusted Entity): 누구에게 위임할 수 있는가


### 권한 정책

- 어떤 권한을 위임할지를 정의하는 yaml 문서
- 예시:
  ```yml
  {
      "Version": "2012-10-17",
      "Statement": [
          {
              "Effect": "Allow",
              "Action": [
                  ...
              ],
              "Resource": "*"
          }
      ]
  }
  ```

#### Effect

#### Action

#### Resource

### 신뢰 대상

- 누구에게 위임할 수 있는지를 지정하는 yaml 문서
- 예시: ec2 서비스에게 위임하는 경우
  ```yml
  {
      "Version": "2012-10-17",
      "Statement": [
          {
              "Effect": "Allow",
              "Principal": {
                  "Service": "ec2.amazonaws.com"
              },
              "Action": "sts:AssumeRole"
          }
      ]
  }
  ```


#### Principal

- Principal: 신뢰할 수 있는 즉, 권한을 부여받을 수 있는 주체에 대한 정의
- Principal이 되려면 반드시 누구인지를 밝히는 신원 정보가 있어야 한다. 
  - 인스턴스, 람다 함수 등은 그 자체로는 신원정보가 없으므로 스스로 권한을 받을 수 없다.
  - 이 경우에는 신뢰 주체가 일단 권한(정확히는 임시 자격증명)을 받아 다시 인스턴스 등으로 넘겨준다. 
- Principal이 될 수 있는 유형
  - AWS service: EC2 서비스, 람다 서비스 등 AWS 자체 서비스는 신뢰 가능
  - AWS account: 계정은 신원이 있으므로 신뢰 가능
  - Web identity
  - SAML 2.0 federation
  - Custom trust policy

#### Action

- Action: 모든 action은 `sts:...`으로 시작
- 모든 권한은 실제로는 STS(Security Token Service) 서비스가 발급하는 임시 자격 증명을 기반으로 사용한다.
- 역할 기반 권한 부여는 다음 단계로 이루어진다.
  1. 신뢰 정책(trust policy)으로 이 역할을 맡을(assume)) 수 있는 주체를 정의한다. (예: ec2 서비스, 람다 서비스 등)
  2. 그 주체가 STS를 통해 역할을 assume하여 임시 자격 증명(액세스 키 ID + 비밀 키 + 보안 토큰)을 발급받는다.
  3. 발급받은 자격 증명을, 실제로 권한을 사용하는 리소스(EC2 인스턴스, Lambda 함수 등)에게 전달한다. 이 전달은 신뢰 정책이 아니라 instance profile 연결 등 별도 메커니즘이 결정한다.